In [13]:
import hashlib
import json
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight


sys.path.insert(0, str(Path.cwd().parent / "src"))

from modifiers.data.preprocessor import StrokePreprocessor
from modifiers.data.loader import StrokeDataLoader
from modifiers.data.augmenter import StrokeAugmenter
from modifiers.features.geometric import GeometricFeatureExtractor
from modifiers.features.renderer import StrokeRenderer
from modifiers.utils.config import load_config
from modifiers.utils.logging import setup_logging, get_logger


In [ ]:

config = load_config("../config/config.yaml")

data_dir = Path.cwd().parent / "data" / "raw"

# Initialize components
loader = StrokeDataLoader(
    data_dir=data_dir,
    valid_classes=config.classes,
)

preprocessor = StrokePreprocessor(normalize=True)

renderer = StrokeRenderer(
    img_size=config.features.image_size,
    line_width=config.features.line_width,
    antialiasing=config.features.antialiasing,
    to_rgb=True,
    normalize_pixels=True,
    scale_factor=4,
    smooth=True,
)

feature_extractor = GeometricFeatureExtractor(
    height_threshold=config.features.height_threshold,
    cap_value=config.features.cap_value,
)

# Load raw data
raw_data = loader.load()

# Preprocess (clean + normalize)
_, processed_data = preprocessor.process_dataset(raw_data, config.class_to_idx)

# Debug: Export Sample Images
debug_dir = output_dir / "debug_samples"
debug_dir.mkdir(parents=True, exist_ok=True)

# Augment if enabled
if config.augmentation.enabled:
    augmenter = StrokeAugmenter(
        num_augmentations=config.augmentation.num_augmentations,
        rotation_range=config.augmentation.rotation_range,
        shear_x_range=config.augmentation.shear_x_range,
        shear_y_range=config.augmentation.shear_y_range,
        scale_x_range=config.augmentation.scale_x_range,
        scale_y_range=config.augmentation.scale_y_range,
        flip_horizontal=config.augmentation.flip_horizontal,
        flip_exclude_types=config.augmentation.flip_exclude_types,
        rotation_exclude_types=config.augmentation.rotation_exclude_types,
        random_seed=config.data.random_state,
    )
    data = augmenter.augment_dataset(processed_data)
else:
    data = processed_data




In [ ]:
df = pd.DataFrame(raw_data)

In [ ]:
def plot_stroke(stroke, ax, title='', color='blue'):
    """Plot a single stroke."""
    x = [p['x'] for p in stroke]
    y = [p['y'] for p in stroke]
    ax.plot(x, y, color=color, linewidth=2)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

# Get unique classes
classes = df['type'].unique()
n_classes = len(classes)

# Plot samples from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

colors = plt.cm.tab10(np.linspace(0, 1, n_classes))

for idx, cls in enumerate(classes[:10]):
    sample = df[df['type'] == cls].iloc[0]
    plot_stroke(sample['stroke'], axes[idx], title=cls, color=colors[idx])

plt.suptitle('Sample Strokes by Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()